[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/08_TreeBasedForecasting.ipynb#copy=true)

# Tree-Based Forecasting and Error Metrics

**Module 0 · Lesson 8 of 13 · Student edition**  
**Estimated class time:** 75–90 minutes  
**Source sequence:** Original Day 3  

**Prerequisite:** Lesson 7  

## Learning objectives

By the end of this lesson, you should be able to:

- Fit random-forest and gradient-boosting forecasters to lag features.
- Compute and interpret RMSE and MAE.
- Compare learned models with naive and AR baselines.

## Setup for this lesson

This cell recreates the data and completed prerequisites from earlier lessons, so this notebook can be run in a fresh kernel.

In [ ]:
# Shared forecasting setup from the preceding lesson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.ar_model import AutoReg
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df.columns = ['Date', 'Passengers']
df['Log_Passengers'] = np.log(df['Passengers'])
df['Log_Diff'] = df['Log_Passengers'].diff()
series = df['Log_Diff'].dropna().values

def make_lag_matrix(series, n_lags):
    series = np.asarray(series)
    X, y = [], []
    for t in range(n_lags, len(series)):
        X.append(series[t - n_lags:t])
        y.append(series[t])
    return np.asarray(X), np.asarray(y)

N_LAGS = 12
X, y = make_lag_matrix(series, N_LAGS)
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

naive_preds = np.concatenate([[y_train[-1]], y_test[:-1]])
train_series = series[:split + N_LAGS]
ar_result = AutoReg(train_series, lags=N_LAGS).fit()
ar_preds = ar_result.predict(
    start=len(train_series),
    end=len(train_series) + len(y_test) - 1,
    dynamic=False,
)


***
## Part 3 — Tree-Based Models

### Introduction to tree-based models

Tree-based models are powerful, non-parametric algorithms that make **no stationarity
assumptions** and can capture non-linear patterns in lag features.

**Random Forest** is an ensemble of $B$ decision trees, each trained on a bootstrap sample
of the training data and a random feature subset. Predictions are averaged:

$$\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(\mathbf{x})$$

**Gradient Boosting** builds trees *sequentially*, where each tree corrects the errors of
the current ensemble. The ensemble is updated as:

$$F_{m}(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot T_m(\mathbf{x})$$

where $\eta$ is the learning rate and $T_m$ fits the residuals of $F_{m-1}$.

> ⚠️ **Important:** Tree-based models are scale-invariant. Do **NOT** scale the lag matrix
> for these models. Scaling is only needed for neural networks (Parts 5–6).

### 3.1 Random Forest

**Your turn!** Fit a `RandomForestRegressor` on the training lag matrix.

> 💡 **Syntax:** `RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)`
> creates the model. `.fit(X, y)` trains it. `.predict(X)` generates predictions.

### Dividir y Confluir 🌊

Work in groups of 2–3. 

| Group | Model | Configuration |
|---|---|---|
| A | Random Forest | `n_estimators=100, max_depth=5` |
| B | Random Forest | `n_estimators=200, max_depth=None` |
| C | Gradient Boosting | `n_estimators=100, learning_rate=0.1` |
| D | Gradient Boosting | `n_estimators=200, learning_rate=0.05` |

Fit your assigned model, compute RMSE and MAE (Part 4), and be ready to share your
results with the class.

In [ ]:
# FILL IN: create and fit a RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=???, max_depth=???, random_state=42)
rf_model.fit(???, ???)
rf_preds = rf_model.predict(???)

print('Random Forest predictions (first 5):', rf_preds[:5])

### 3.2 Gradient Boosting

In [ ]:
# FILL IN: create and fit a GradientBoostingRegressor
gb_model = GradientBoostingRegressor(n_estimators=???, learning_rate=???, random_state=42)
gb_model.fit(???, ???)
gb_preds = gb_model.predict(???)

print('Gradient Boosting predictions (first 5):', gb_preds[:5])

***
## Part 4 — Error Metrics & First Comparison

### Two standard metrics

Let $y_t$ be the actual value and $\hat{y}_t$ the model prediction at test step $t$.
For a test set of size $n$:

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{t=1}^{n} (y_t - \hat{y}_t)^2}$$

$$\text{MAE}  = \frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t|$$

**RMSE** penalizes large errors more heavily (via squaring). It is preferred when big
mistakes are disproportionately costly.

**MAE** treats all errors equally and is more robust to outliers. It is preferred when
a consistent, interpretable average error is desired.

### 4.1 Implement the metrics function

**Your turn!** Complete the `compute_metrics` function below.

In [ ]:
def compute_metrics(y_true, y_pred, label='Model'):
    """
    Compute and print RMSE and MAE for a set of predictions.

    Parameters
    ----------
    y_true : array-like, actual values
    y_pred : array-like, predicted values
    label  : str, model name for display

    Returns
    -------
    dict with keys 'RMSE' and 'MAE'
    """
    # FILL IN: compute RMSE (hint: np.sqrt + mean_squared_error)
    rmse = ???
    # FILL IN: compute MAE
    mae  = ???
    print(f'{label:<30}  RMSE={rmse:.5f}   MAE={mae:.5f}')
    return {'RMSE': rmse, 'MAE': mae}


# Evaluate the first four models
results = {}
results['Naive Baseline']    = compute_metrics(y_test, naive_preds,      'Naive Baseline')
results['AR(12)']            = compute_metrics(y_test, ar_preds,  'AR(12)')
results['Random Forest']     = compute_metrics(y_test, rf_preds,         'Random Forest')
results['Gradient Boosting'] = compute_metrics(y_test, gb_preds,         'Gradient Boosting')

In [ ]:
# Visualise predictions on the test set
fig, ax = plt.subplots(figsize=(14, 5))
test_range = np.arange(len(y_test))

ax.plot(test_range, y_test,          label='Actual',           color='black',     linewidth=2)
ax.plot(test_range, naive_preds,     label='Naive Baseline',   color='gray',      linestyle='--')
ax.plot(test_range, ar_preds,        label='AR(12)',           color='steelblue', linestyle='-.')
ax.plot(test_range, rf_preds,        label='Random Forest',    color='forestgreen')
ax.plot(test_range, gb_preds,        label='Gradient Boosting',color='darkorange')

ax.set_title('Test Set Predictions — Log-Differenced Passengers')
ax.set_xlabel('Test Step')
ax.set_ylabel('Log-Differenced Value')
ax.legend()
plt.tight_layout()
plt.show()

### ✏️ Written Response 4

Based on your RMSE and MAE values, answer in 3–5 sentences:

1. Which model family performs best so far? Does this surprise you? Why or why not?
2. Do the RMSE and MAE rankings agree? If one metric suggested a different winner than
   the other, what would that imply about the error distribution?
3. Why is it essential to compare against the naive baseline rather than reporting
   model metrics in isolation?

**After completing your analysis, share your results with the class** (dividir y confluir
debrief). Were the rankings consistent across groups, or did different hyperparameters
produce different rankings?

> **YOUR ANSWER:**